# Proving Audio Frequency Visualization Works

Our extension uses the Web Audio API's `AnalyserNode` to generate a frequency visualization. This notebook proves the underlying mathematics (Fast Fourier Transform or FFT) that powers that visualizer.

We will generate synthetic audio (combining a low and high-frequency wave) and show how the FFT correctly separates them into distinct visual bars, exactly like the browser extension does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup sampling rate and time axis
sample_rate = 44100  # Standard audio sampling rate
duration = 1.0       # 1 second of audio
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

### Generate the Audio Signal
Let's create an audio signal made of two distinct frequencies: 440 Hz (A4) and 4000 Hz.

In [ ]:
# Generate a low frequency (440 Hz) and a high frequency (4000 Hz) wave
freq1 = 440.0
freq2 = 4000.0

wave1 = 0.5 * np.sin(2 * np.pi * freq1 * t)
wave2 = 0.3 * np.sin(2 * np.pi * freq2 * t)

# Combine them
audio_signal = wave1 + wave2

# Plot a small segment of the raw audio waveform
plt.figure(figsize=(10, 3))
plt.plot(t[:1000], audio_signal[:1000])
plt.title("Raw Audio Waveform (Time Domain)")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()

### Perform Fast Fourier Transform (FFT)
This is the mathematical equivalent of `AnalyserNode.getByteFrequencyData` in JavaScript.

In [ ]:
# Compute FFT
fft_result = np.fft.fft(audio_signal)
fft_magnitude = np.abs(fft_result)

# Get frequencies corresponding to the FFT values
frequencies = np.fft.fftfreq(len(fft_result), 1/sample_rate)

# Only take the positive half of the spectrum
half_n = len(frequencies) // 2
frequencies = frequencies[:half_n]
fft_magnitude = fft_magnitude[:half_n]

# Plot the Frequency Domain (Visualizer Bars)
plt.figure(figsize=(10, 4))
plt.plot(frequencies, fft_magnitude)
plt.title("Frequency Spectrum (What the Extension Draws)")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.xlim(0, 5000)  # Focus on the relevant range up to 5kHz
plt.grid(True)
plt.show()

### Conclusion
As we can see, the FFT successfully identified spikes at exactly 440 Hz and 4000 Hz. When the user speaks into their microphone, the browser performs this math rapidly in chunks, generating the exact data used to draw the dynamic bar graph in our floating UI!